In [13]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('Working with files').getOrCreate()

In [14]:
from google.colab import files

uploaded = files.upload()

In [15]:
df = spark.read.csv(
    'flightBookings.csv',
    header = True,
    inferSchema = True
)

df.show()

+----------+--------------+---------+---------+------------------+------------+------------+---------+
|booking_id|passenger_name|from_city|  to_city|           airline|ticket_price|travel_class|   status|
+----------+--------------+---------+---------+------------------+------------+------------+---------+
|      1001|   Aarav Mehta|Hyderabad|    Delhi|            IndiGo|        6500|     Economy|Confirmed|
|      1002|     Sana Khan|Bangalore|   Mumbai|           Vistara|        8200|     Economy|Confirmed|
|      1003|   John Mathew|  Chennai|    Delhi|         Air India|       12000|    Business|Confirmed|
|      1004|  Ayesha Begum|Hyderabad|    Dubai|          Emirates|       28000|     Economy|Confirmed|
|      1005|    Vikram Rao|   Mumbai|Singapore|Singapore Airlines|       35000|    Business|  Pending|
|      1006|  Divya Sharma|    Delhi|Hyderabad|            IndiGo|        5900|     Economy|Cancelled|
|      1007|     Imran Ali|     Pune|Bangalore|         Akasa Air|       

practice

In [16]:
df.printSchema()

root
 |-- booking_id: integer (nullable = true)
 |-- passenger_name: string (nullable = true)
 |-- from_city: string (nullable = true)
 |-- to_city: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- ticket_price: integer (nullable = true)
 |-- travel_class: string (nullable = true)
 |-- status: string (nullable = true)



In [17]:
from pyspark.sql.functions import max, min, sum, avg, col, when

df.show()
df.select('passenger_name', 'from_city', 'to_city').show()
df.filter(df.status == 'Confirmed').show()
df.filter(df.status == 'Cancelled').show()
df.filter(df.status == 'Pending').show()
df.filter(df.ticket_price > 20000).show()
df.filter(col('to_city').isin('Dubai', 'Singapore', 'London')).show()
print('total booking: ', df.count())
df.groupby('airline').count().show()
df.groupby('status').count().show()
df.filter(df.status == 'Confirmed').select(sum('ticket_price')).show()
df.groupby('airline').avg('ticket_price').show()
df.groupby('airline').max('ticket_price').show()
df.groupby('airline').min('ticket_price').show()
df.orderBy(col('ticket_price').desc()).show()

full_col = df.withColumn(
    'tax',
    col('ticket_price') * 0.05
)

full_col = full_col.withColumn(
    'final_price',
    col('ticket_price') + col('tax')
)

full_col = full_col.withColumn(
    'price_category',
    when(col('ticket_price') >= 30000, 'Premium')
    .when(col('ticket_price') >= 10000, 'Standard')
    .otherwise('Budget')
)

full_col.show()
full_col.groupby('price_category').count().show()
full_col.filter(full_col.status == 'Confirmed').write.mode('overwrite').parquet('confirmed_flights.parquet')

+----------+--------------+---------+---------+------------------+------------+------------+---------+
|booking_id|passenger_name|from_city|  to_city|           airline|ticket_price|travel_class|   status|
+----------+--------------+---------+---------+------------------+------------+------------+---------+
|      1001|   Aarav Mehta|Hyderabad|    Delhi|            IndiGo|        6500|     Economy|Confirmed|
|      1002|     Sana Khan|Bangalore|   Mumbai|           Vistara|        8200|     Economy|Confirmed|
|      1003|   John Mathew|  Chennai|    Delhi|         Air India|       12000|    Business|Confirmed|
|      1004|  Ayesha Begum|Hyderabad|    Dubai|          Emirates|       28000|     Economy|Confirmed|
|      1005|    Vikram Rao|   Mumbai|Singapore|Singapore Airlines|       35000|    Business|  Pending|
|      1006|  Divya Sharma|    Delhi|Hyderabad|            IndiGo|        5900|     Economy|Cancelled|
|      1007|     Imran Ali|     Pune|Bangalore|         Akasa Air|       

In [8]:
uploaded_02 = files.upload()

Saving hotels.json to hotels.json


In [18]:
hotels_df = spark.read.option(
    'multiline',
    'true'
).json('hotels.json')
hotels_df.show(truncate=False)
hotels_df.printSchema()

+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|amenities                  |category|city     |contact                          |hotel_id|hotel_name      |price_per_night|rating|rooms_available|
+---------------------------+--------+---------+---------------------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast, gym]     |Business|Hyderabad|{pearlgrand@mail.com, 9876500011}|201     |Pearl Grand     |4500           |4.4   |25             |
|[wifi, pool, spa, sea_view]|Luxury  |Dubai    |{marinabay@mail.com, 9876500012} |202     |Marina Bay Stay |18000          |4.8   |12             |
|[wifi]                     |Budget  |Delhi    |{budgetinn@mail.com, NULL}       |203     |Budget Inn      |2200           |3.9   |40             |
|[wifi, breakfast, pool]    |Resort  |Kochi    |{NULL, 9876500014}               |204     |Hill View Resort|7500

In [19]:
from pyspark.sql.functions import max, min, sum, avg, col, when, array_contains

hotels_df.show()
hotels_df.select('hotel_name', 'city', 'rating').show()
hotels_df.filter(hotels_df.category == 'Luxury').show()
hotels_df.filter(hotels_df.rating > 4.5).show()
hotels_df.filter(hotels_df.rooms_available > 15).show()
hotels_df.filter(hotels_df.price_per_night > 10000).show()
hotels_df.filter(hotels_df.city.isin('Dubai', 'London')).show()
hotels_df.filter(hotels_df.contact.phone.isNull()).show()
hotels_df.filter(hotels_df.contact.email.isNull()).show()
hotels_df.select('hotel_name', 'contact.phone', 'contact.email').show()
hotels_df.filter(array_contains(col('amenities'), 'wifi')).show()
hotels_df.filter(array_contains(col('amenities'), 'spa')).show()
hotels_df.groupby('city').count().show()
hotels_df.groupby('category').count().show()
hotels_df.select(avg('rating').alias('avg rating')).show()
hotels_df.select(avg('price_per_night').alias('avg price')).show()
hotels_df.select(max('price_per_night').alias('max price')).show()

full_hotels_df = hotels_df.withColumn(
    'total_potential_revenue',
    col('rooms_available') * col('price_per_night')
)
full_hotels_df.show()

hotels_df.orderBy(col('rating').desc()).show()

cols = [x for x in hotels_df.columns if x != 'contact']
flatten_hotels_df = hotels_df.select(
    *cols,
    col('contact.phone').alias('phone'),
    col('contact.email').alias('email')
)
flatten_hotels_df.show()

flatten_hotels_df.write.mode('overwrite').parquet('hotels_flattened.parquet')

+--------------------+--------+---------+--------------------+--------+----------------+---------------+------+---------------+
|           amenities|category|     city|             contact|hotel_id|      hotel_name|price_per_night|rating|rooms_available|
+--------------------+--------+---------+--------------------+--------+----------------+---------------+------+---------------+
|[wifi, breakfast,...|Business|Hyderabad|{pearlgrand@mail....|     201|     Pearl Grand|           4500|   4.4|             25|
|[wifi, pool, spa,...|  Luxury|    Dubai|{marinabay@mail.c...|     202| Marina Bay Stay|          18000|   4.8|             12|
|              [wifi]|  Budget|    Delhi|{budgetinn@mail.c...|     203|      Budget Inn|           2200|   3.9|             40|
|[wifi, breakfast,...|  Resort|    Kochi|  {NULL, 9876500014}|     204|Hill View Resort|           7500|   4.5|             18|
|[wifi, breakfast,...|  Luxury|   London|{skyline@mail.com...|     205|  Skyline Suites|          22000|